In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Dense,Attention

# Define an input layer
input_layer = Input(shape=(sequence_length, input_dimension))

# Add an attention layer
attention_layer = Attention()([input_layer, input_layer])

# Add other layers as needed
# ...

# Compile and train the model
model = keras.Model(inputs=input_layer, outputs=output_layer)
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(x_train, y_train, epochs=num_epochs, batch_size=batch_size)



import tensorflow as tf
import numpy as np

# Define the input and target language data
input_data = ["I love deep learning", "Machine translation is fascinating", "Attention mechanisms improve models"]
target_data = ["J'adore l'apprentissage profond", "La traduction automatique est fascinante", "Les mécanismes d'attention améliorent les modèles"]

# Tokenize the data
input_tokenizer = tf.keras.layers.TextVectorization()
input_tokenizer.adapt(input_data)
target_tokenizer = tf.keras.layers.TextVectorization()
target_tokenizer.adapt(target_data)

# Create tokenized sequences
input_sequences = input_tokenizer(input_data)
target_sequences = target_tokenizer(target_data)

# Define the model architecture
embedding_dim = 256
units = 1024

# Encoder
encoder_inputs = tf.keras.layers.Input(shape=(None,))
encoder_embedding = tf.keras.layers.Embedding(input_tokenizer.vocabulary_size, embedding_dim)(encoder_inputs)
encoder, encoder_state_h, encoder_state_c = tf.keras.layers.LSTM(units, return_state=True)(encoder_embedding)
encoder_states = [encoder_state_h, encoder_state_c]

# Decoder
decoder_inputs = tf.keras.layers.Input(shape=(None,))
decoder_embedding = tf.keras.layers.Embedding(target_tokenizer.vocabulary_size, embedding_dim)(decoder_inputs)
decoder_lstm = tf.keras.layers.LSTM(units, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
attention = tf.keras.layers.Attention()([decoder_outputs, encoder_outputs])
decoder_concat = tf.keras.layers.Concatenate(axis=-1)([decoder_outputs, attention])
decoder_dense = tf.keras.layers.Dense(target_tokenizer.vocabulary_size, activation='softmax')(decoder_concat)

# Create the model
model = tf.keras.models.Model([encoder_inputs, decoder_inputs], decoder_dense)

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model (you would need a larger dataset for real training)
model.fit([input_sequences, target_sequences[:, :-1]], target_sequences[:, 1:], epochs=50)

# Translate a sample sentence
def translate(input_text):
    input_seq = input_tokenizer([input_text])
    target_seq = np.zeros((1, target_max_length))
    target_seq[0, 0] = target_tokenizer.word_index['<start>']
    
    for i in range(1, target_max_length):
        predicted = model.predict([input_seq, target_seq])
        predicted_word_index = np.argmax(predicted[:, i-1, :])
        target_seq[0, i] = predicted_word_index
        
        if target_tokenizer.index_word[predicted_word_index] == '<end>':
            break
    
    translated_sentence = ' '.join([target_tokenizer.index_word[i] for i in target_seq[0] if i not in [0]])
    return translated_sentence

# Example translation
input_text = "I love deep learning"
translated_text = translate(input_text)
print("Input:", input_text)
print("Translation:", translated_text)
